# PCA Framework: Testing Hypotheses on VTA Dopamine & GABA Dynamics

This notebook implements a systematic analytical framework to test whether VTA neural population dynamics encode **reward value** (RPE theory) or **movement direction** (kinematic hypothesis).

## Hypotheses

- **H1**: Movement direction dominates latent structure — SpontFB PCA basis generalises to task data, and CRFB/ToneFB share a common subspace
- **H2**: DA and GABA populations encode different latent variables — GABA is low-dimensional and direction-selective; DA is heterogeneous and task-modulated
- **H3**: GABA (and DA) trajectories do **not** show a reward-specific deflection at reward delivery time
- **H4**: CS-evoked and CR-evoked phasic DA are the same movement-related burst, time-shifted by reaction time — not genuinely different dynamics
- **H5**: Aversive stimuli test the value axis (FUTURE — awaiting airpuff data)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import importlib

import plot_pca_framework
importlib.reload(plot_pca_framework)

from plot_pca_framework import (
    load_dataset,
    extract_neuron_data,
    extract_group_averaged_data,
    fit_pca,
    project_onto_pca,
    align_pca_signs,
    run_pca,
    slice_epoch,
    slice_window,
    smooth_trajectories,
    get_event_markers,
    analyze_dataset,
    cross_project,
    cross_class_project,
    compute_trajectory_metrics,
    compute_reconstruction_r2,
    compute_subspace_overlap,
    compute_procrustes_distance,
    compute_cross_correlation,
    compute_cross_epoch_r2_matrix,
    plot_1d_pc_timecourses,
    build_overlay_figure,
    plot_scree_comparison,
    plot_speed_profiles,
    plot_cross_epoch_r2_matrix,
    plot_metric_comparison_table,
    EPOCHS,
    # Epoch-specific analysis
    analyze_epoch,
    get_epoch_event_markers,
    save_epoch_trajectories,
    # Analysis functions
    compute_participation_ratio,
    compute_divergence_onset,
    compute_pc_loadings_by_group,
    plot_pc_loadings,
    plot_participation_ratio_comparison,
    plot_divergence_comparison,
    # RSA + Procrustes
    compute_rdm,
    compare_rdms,
    compute_rsa,
    compute_procrustes_comparison,
    plot_rdm,
    plot_rsa_comparison,
    plot_procrustes_comparison,
    # Null models
    null_cross_projection_r2,
    null_separation,
    null_cross_class_r2,
    null_same_neuron_cross_r2,
    null_reward_deflection,
)

logging.basicConfig(level=logging.WARNING)
%matplotlib inline
plt.rcParams["figure.dpi"] = 50

## Configuration

In [ ]:
# Dataset definitions
DATASETS = {
    'SpontFB': {'mat_file': 'dataSpontFB.mat', 'var_name': 'dataSpontFB'},
    'CRFB':    {'mat_file': 'dataCRFB.mat',    'var_name': 'dataCRFB'},
    'ToneFB':  {'mat_file': 'dataToneFB.mat',  'var_name': 'dataToneFB'},
}

# Neuron groups
DA_GROUPS = ['DF', 'DB', 'D', 'DFB']
GABA_GROUPS = ['GF', 'GB', 'G', 'GFB']
ALL_GROUPS = DA_GROUPS + GABA_GROUPS

# Analysis parameters
N_COMPONENTS = 3
EVENT_IDX = 600
WINDOW = 150  # expanded from 120 to capture post-reward dynamics
DT = 0.01
SG_WINDOW = 11
SG_ORDER = 3

## Step 0: Load All Datasets & Run PCA

Run all 9 analyses (3 datasets x {Dopamine, GABA, Combined}). The DFB group has NaN rows in CRFB/ToneFB (rows 19 and 65); these are dropped automatically.

In [ ]:
# Run all 9 dataset x neuron-combo analyses
results = {}
combos = {'Dopamine': DA_GROUPS, 'GABA': GABA_GROUPS, 'Combined': ALL_GROUPS}

for ds_name, ds_cfg in DATASETS.items():
    for combo_name, groups in combos.items():
        key = f"{ds_name}_{combo_name}"
        try:
            r = analyze_dataset(
                mat_file=ds_cfg['mat_file'],
                var_name=ds_cfg['var_name'],
                dataset_name=ds_name,
                neuron_groups=groups,
                combo_label=combo_name,
                n_components=N_COMPONENTS,
                event_idx=EVENT_IDX,
                window=WINDOW,
                dt=DT,
                sg_window=SG_WINDOW,
                sg_order=SG_ORDER,
            )
            results[key] = r
            evr = r['explained_variance_ratio']
            print(f"OK  {key:30s}  n={r['n_neurons']:4d}  "
                  f"EVR=[{evr[0]:.3f}, {evr[1]:.3f}, {evr[2]:.3f}]  "
                  f"total={sum(evr):.3f}")
            for g, s in r['stats'].items():
                if s['dropped'] > 0:
                    print(f"     {g}: dropped {s['dropped']}/{s['total']} neurons (NaN)")
        except Exception as e:
            print(f"FAIL {key:30s}  {e}")

print(f"\nSuccessful: {len(results)}/9")

### Step 0 — Data Summary

**All 9/9 analyses succeed.** NaN fix works: DFB drops 1-2 neurons in CRFB/ToneFB.

**Key EVR observations:**
- **GABA is lower-dimensional than DA:** PC1 captures 27-39% (GABA) vs 7-21% (DA). GABA population activity is more coordinated / low-rank; DA is heterogeneous.
- **ToneFB DA** has unusually high PC1 variance (20.9%) compared to CRFB DA (8.8%) and SpontFB DA (7.0%). The tone/CS synchronises DA population dynamics more strongly than CR-onset or spontaneous movement.
- **Combined** (DA+GABA) EVR lies between pure-DA and pure-GABA, as expected for mixed populations.

---
# H1: Movement Direction Dominates VTA Latent Structure

**Question:** Is movement direction the primary organiser of VTA population dynamics, such that PCA bases learned from one behavioral context generalise to others?

**Prediction:** If movement direction dominates, then:
1. Cross-projecting CRFB onto ToneFB PCA (same neurons, different alignment) should yield high subspace overlap
2. Group-averaged cross-projection from SpontFB onto task data (different neurons) should yield high R-squared
3. Both should significantly exceed null models

**Tests ordered by stringency:**
- H1.1: Same-neuron cross-projection (CRFB <-> ToneFB) — strongest, same neurons
- H1.2: Group-averaged cross-projection (SpontFB <-> Task) — different neurons
- H1.3-H1.5: Overlays, RSA/Procrustes, null models

### H1.1: Same-Neuron Cross-Projection — CRFB <-> ToneFB

CRFB and ToneFB contain the **exact same neurons** aligned to different events (movement onset vs tone onset). Fit PCA on one dataset, project the other, compute R-squared and subspace overlap.

In [ ]:
# Same-neuron cross-projection: GABA
cross_results_same_neuron = {}

if 'ToneFB_GABA' in results and 'CRFB_GABA' in results:
    cross_results_same_neuron['GABA Tone->CR'] = cross_project(
        results['ToneFB_GABA'], results['CRFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    cross_results_same_neuron['GABA CR->Tone'] = cross_project(
        results['CRFB_GABA'], results['ToneFB_GABA'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    for name, cr in cross_results_same_neuron.items():
        print(f"{name}:  R-squared = {cr['r2']:.4f}")

    overlap_gaba = compute_subspace_overlap(
        results['ToneFB_GABA']['pca'].components_,
        results['CRFB_GABA']['pca'].components_,
    )
    print(f"\nGABA subspace overlap (cos principal angles): {overlap_gaba}")
else:
    print('ToneFB_GABA or CRFB_GABA not available')

# Same-neuron cross-projection: DA
if 'ToneFB_Dopamine' in results and 'CRFB_Dopamine' in results:
    cross_results_same_neuron['DA Tone->CR'] = cross_project(
        results['ToneFB_Dopamine'], results['CRFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    cross_results_same_neuron['DA CR->Tone'] = cross_project(
        results['CRFB_Dopamine'], results['ToneFB_Dopamine'],
        use_group_avg=False, window=WINDOW, event_idx=EVENT_IDX,
        dt=DT, sg_window=SG_WINDOW, sg_order=SG_ORDER,
    )
    for name, cr in cross_results_same_neuron.items():
        if 'DA' in name:
            print(f"{name}:  R-squared = {cr['r2']:.4f}")

    overlap_da = compute_subspace_overlap(
        results['ToneFB_Dopamine']['pca'].components_,
        results['CRFB_Dopamine']['pca'].components_,
    )
    print(f"\nDA subspace overlap (cos principal angles): {overlap_da}")
else:
    print('ToneFB_Dopamine or CRFB_Dopamine not available')

### H1.2: Group-Averaged Cross-Projection — SpontFB <-> Task Data

SpontFB uses different individual neurons than CRFB/ToneFB, so direct cross-projection is impossible. Instead, average firing rates within each neuron group (DF, DB, D, DFB / GF, GB, G, GFB) to create 4 "pseudo-neurons," then cross-project.

**Caveat:** With 4 features and 3 PCs, R-squared is structurally biased upward. The null model (H1.5) addresses this.

In [ ]:
# GABA group-averaged cross-projection: SpontFB <-> Task
cross_results_group_avg = {}

for target_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if 'SpontFB_GABA' in results and target_name in results:
        key = f'Spont->{target_name.split("_")[0]}'
        cross_results_group_avg[key] = cross_project(
            results['SpontFB_GABA'], results[target_name],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R-squared = {cross_results_group_avg[key]['r2']:.4f}")

for source_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if source_name in results and 'SpontFB_GABA' in results:
        key = f'{source_name.split("_")[0]}->Spont'
        cross_results_group_avg[key] = cross_project(
            results[source_name], results['SpontFB_GABA'],
            use_group_avg=True, neuron_groups=GABA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        print(f"GABA {key}: R-squared = {cross_results_group_avg[key]['r2']:.4f}")

In [ ]:
# DA group-averaged cross-projection: SpontFB <-> Task
for target_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if 'SpontFB_Dopamine' in results and target_name in results:
        key = f'DA Spont->{target_name.split("_")[0]}'
        cr = cross_project(
            results['SpontFB_Dopamine'], results[target_name],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_group_avg[key] = cr
        print(f"{key}: R-squared = {cr['r2']:.4f}")

for source_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if source_name in results and 'SpontFB_Dopamine' in results:
        key = f'DA {source_name.split("_")[0]}->Spont'
        cr = cross_project(
            results[source_name], results['SpontFB_Dopamine'],
            use_group_avg=True, neuron_groups=DA_GROUPS,
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        cross_results_group_avg[key] = cr
        print(f"{key}: R-squared = {cr['r2']:.4f}")

### H1.3: Overlay Visualisations

In [ ]:
# Overlay: ToneFB native + CRFB projected (GABA)
if 'GABA Tone->CR' in cross_results_same_neuron and 'ToneFB_GABA' in results:
    tsets = [
        {
            'fwd_smooth': results['ToneFB_GABA']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_GABA']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['ToneFB_GABA']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_same_neuron['GABA Tone->CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_same_neuron['GABA Tone->CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB (projected)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': cross_results_same_neuron['GABA Tone->CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H1: ToneFB vs CRFB GABA (ToneFB PC space)')
    fig_overlay.show()

In [ ]:
# Overlay: SpontFB native + Task projected (GABA, group-averaged)
if 'SpontFB_GABA' in results:
    tsets = [
        {
            'fwd_smooth': results['SpontFB_GABA']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['SpontFB_GABA']['smooth_data']['bwd_smooth'],
            'label': 'SpontFB GABA (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['SpontFB_GABA']['event_markers'],
        },
    ]
    for key, color_fwd, color_bwd in [
        ('Spont->CRFB', 'gold', 'teal'),
        ('Spont->ToneFB', 'lime', 'purple'),
    ]:
        if key in cross_results_group_avg:
            cr = cross_results_group_avg[key]
            tsets.append({
                'fwd_smooth': cr['smooth_data']['fwd_smooth'],
                'bwd_smooth': cr['smooth_data']['bwd_smooth'],
                'label': f'{cr["project_dataset"]} GABA (proj)',
                'fwd_color': color_fwd, 'bwd_color': color_bwd, 'dash': 'dash',
                'event_markers': cr['event_markers'],
            })
    if len(tsets) > 1:
        fig = build_overlay_figure(
            tsets, title='H1: SpontFB vs Task GABA (group-avg SpontFB PC space)')
        fig.show()

### H1.4: RSA + Procrustes — SpontFB vs Task Data

RSA compares pairwise temporal distance matrices (basis-free). Procrustes compares trajectory shape (rotation-invariant). Both computed on full-space AND 3-PC projections for completeness.

In [ ]:
# H1.4: RSA + Procrustes -- SpontFB vs Task
rsa_h1_results = {}
procrustes_h1_results = {}

for combo_name in ['GABA', 'Dopamine']:
    spont_key = f'SpontFB_{combo_name}'
    if spont_key not in results:
        continue
    X_spont = results[spont_key]['X']
    sd_spont = results[spont_key]['smooth_data']
    n_t = X_spont.shape[1] // 2

    for target_ds in ['CRFB', 'ToneFB']:
        target_key = f'{target_ds}_{combo_name}'
        if target_key not in results:
            continue
        X_target = results[target_key]['X']
        sd_target = results[target_key]['smooth_data']

        # Full-space RSA on fwd and bwd halves
        for half_name, sl in [('fwd', slice(0, n_t)), ('bwd', slice(n_t, 2*n_t))]:
            rdm_spont = compute_rdm(X_spont.values[:, sl])
            rdm_target = compute_rdm(X_target.values[:, sl])
            r_val = compare_rdms(rdm_spont, rdm_target, method='cosine')
            label = f'{combo_name}: Spont-{target_ds}_{half_name} (full-space)'
            rsa_h1_results[label] = {'r': r_val, 'p': float('nan')}
            print(f"  RSA {label}: similarity={r_val:.4f}")

        # 3-PC RSA
        proj_spont = results[spont_key]['projections']
        proj_target = results[target_key]['projections']
        for half_name, sl in [('fwd', slice(0, n_t)), ('bwd', slice(n_t, 2*n_t))]:
            rdm_spont_pc = compute_rdm(proj_spont[:, sl])
            rdm_target_pc = compute_rdm(proj_target[:, sl])
            r_val = compare_rdms(rdm_spont_pc, rdm_target_pc, method='cosine')
            label = f'{combo_name}: Spont-{target_ds}_{half_name} (3-PC)'
            rsa_h1_results[label] = {'r': r_val, 'p': float('nan')}
            print(f"  RSA {label}: similarity={r_val:.4f}")

        # Procrustes on smoothed trajectories
        proc = compute_procrustes_comparison(sd_spont, sd_target)
        proc_label = f'{combo_name}: Spont-{target_ds}'
        procrustes_h1_results[proc_label] = proc
        for d, v in proc.items():
            print(f"  Procrustes {proc_label} {d}: disparity={v['disparity']:.4f}")

if rsa_h1_results:
    plot_rsa_comparison(rsa_h1_results, title='H1.4: RSA -- SpontFB vs Task')
if procrustes_h1_results:
    flat_proc = {}
    for lbl, dirs in procrustes_h1_results.items():
        for d, v in dirs.items():
            flat_proc[f'{lbl}_{d}'] = v
    plot_procrustes_comparison(flat_proc, title='H1.4: Procrustes -- SpontFB vs Task')

### H1.5: Null Models

**Group-averaged null (phase randomisation):** Independently phase-randomise each group's time-course in the projected dataset. Preserves autocorrelation and power spectrum; destroys cross-group temporal alignment. Tests whether R-squared > 0.9 is genuine or a structural artefact of 4 features / 3 PCs.

**Same-neuron null (circular time-shift):** Independently circularly shift each neuron's timecourse in the fitting dataset. Preserves per-neuron autocorrelation; destroys inter-neuron temporal alignment. Tests whether CRFB<->ToneFB R-squared reflects genuine shared covariance.

In [ ]:
# Null model: GABA group-averaged cross-projection R-squared
null_h1_gaba = {}
for target_name in ['CRFB_GABA', 'ToneFB_GABA']:
    if 'SpontFB_GABA' not in results or target_name not in results:
        continue
    key = f'Spont->{target_name.split("_")[0]}'
    try:
        obs_r2, null_r2s, p_val = null_cross_projection_r2(
            results['SpontFB_GABA']['data'],
            results[target_name]['data'],
            neuron_groups=GABA_GROUPS,
            n_components=N_COMPONENTS,
            n_permutations=500, seed=42,
        )
        null_h1_gaba[key] = {'observed': obs_r2, 'null': null_r2s, 'p': p_val}
        effect_size = (obs_r2 - np.mean(null_r2s)) / (np.std(null_r2s) + 1e-12)
        print(f'GABA {key}:  R2={obs_r2:.4f}  null median={np.median(null_r2s):.4f}'
              f'  p={p_val:.4f}  effect_size={effect_size:.1f}')
    except Exception as e:
        print(f'GABA {key}: null model failed - {e}')

if null_h1_gaba:
    fig, axes = plt.subplots(1, len(null_h1_gaba), figsize=(6*len(null_h1_gaba), 4))
    if len(null_h1_gaba) == 1: axes = [axes]
    for ax, (k, res) in zip(axes, null_h1_gaba.items()):
        ax.hist(res['null'], bins=30, alpha=0.7, color='grey', label='Null')
        ax.axvline(res['observed'], color='red', lw=2,
                   label=f'Observed ({res["observed"]:.3f})')
        ax.set_title(f'GABA {k}\np = {res["p"]:.4f}')
        ax.set_xlabel('R-squared'); ax.legend()
    fig.suptitle('Null: GABA Group-Avg Cross-Projection R-squared', fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Null model: DA group-averaged cross-projection R-squared
null_h1_da = {}
for target_name in ['CRFB_Dopamine', 'ToneFB_Dopamine']:
    if 'SpontFB_Dopamine' not in results or target_name not in results:
        continue
    key = f'DA Spont->{target_name.split("_")[0]}'
    try:
        obs_r2, null_r2s, p_val = null_cross_projection_r2(
            results['SpontFB_Dopamine']['data'],
            results[target_name]['data'],
            neuron_groups=DA_GROUPS,
            n_components=N_COMPONENTS,
            n_permutations=500, seed=42,
        )
        null_h1_da[key] = {'observed': obs_r2, 'null': null_r2s, 'p': p_val}
        effect_size = (obs_r2 - np.mean(null_r2s)) / (np.std(null_r2s) + 1e-12)
        print(f'{key}:  R2={obs_r2:.4f}  null median={np.median(null_r2s):.4f}'
              f'  p={p_val:.4f}  effect_size={effect_size:.1f}')
    except Exception as e:
        print(f'{key}: null model failed - {e}')

if null_h1_da:
    fig, axes = plt.subplots(1, len(null_h1_da), figsize=(6*len(null_h1_da), 4))
    if len(null_h1_da) == 1: axes = [axes]
    for ax, (k, res) in zip(axes, null_h1_da.items()):
        ax.hist(res['null'], bins=30, alpha=0.7, color='grey', label='Null')
        ax.axvline(res['observed'], color='red', lw=2,
                   label=f'Observed ({res["observed"]:.3f})')
        ax.set_title(f'{k}\np = {res["p"]:.4f}')
        ax.set_xlabel('R-squared'); ax.legend()
    fig.suptitle('Null: DA Group-Avg Cross-Projection R-squared', fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Null model: Same-neuron cross-projection R-squared (CRFB <-> ToneFB)
null_h1_same = {}
for pop_name, pop_label in [('GABA', 'GABA'), ('Dopamine', 'DA')]:
    tone_key = f'ToneFB_{pop_name}'
    cr_key = f'CRFB_{pop_name}'
    if tone_key not in results or cr_key not in results:
        continue
    try:
        obs_r2, null_r2s, p_val = null_same_neuron_cross_r2(
            results[tone_key], results[cr_key],
            n_components=N_COMPONENTS, n_permutations=500, seed=42,
        )
        null_h1_same[f'{pop_label} Tone->CR'] = {
            'observed': obs_r2, 'null': null_r2s, 'p': p_val}
        effect_size = (obs_r2 - np.mean(null_r2s)) / (np.std(null_r2s) + 1e-12)
        print(f'{pop_label} Tone->CR:  R2={obs_r2:.4f}  '
              f'null median={np.median(null_r2s):.4f}  '
              f'p={p_val:.4f}  effect_size={effect_size:.1f}')
    except Exception as e:
        print(f'{pop_label} same-neuron null failed - {e}')

if null_h1_same:
    fig, axes = plt.subplots(1, len(null_h1_same), figsize=(6*len(null_h1_same), 4))
    if len(null_h1_same) == 1: axes = [axes]
    for ax, (k, res) in zip(axes, null_h1_same.items()):
        ax.hist(res['null'], bins=30, alpha=0.7, color='grey', label='Null')
        ax.axvline(res['observed'], color='red', lw=2,
                   label=f'Observed ({res["observed"]:.3f})')
        ax.set_title(f'{k}\np = {res["p"]:.4f}')
        ax.set_xlabel('R-squared'); ax.legend()
    fig.suptitle('Null: Same-Neuron Cross-Projection R-squared (circular shift)',
                 fontweight='bold')
    plt.tight_layout(); plt.show()

### H1 — Interpretation

**Same-neuron cross-projection (CRFB <-> ToneFB):**
- GABA subspace overlap near 1.0: ToneFB and CRFB GABA span essentially the same 3D subspace. The axes of population variability are preserved across alignment conditions.
- GABA R-squared ~0.40-0.45: the subspace is shared, but only ~40% of temporal variance is reconstructed. The SAME latent axes are used, but the temporal dynamics differ — CS-onset and CR-onset modulate the same population modes with different temporal profiles.
- DA R-squared is much lower (~0.08-0.18) and subspace overlap is partial. DA dynamics are more timing-sensitive; CS and CR activate different temporal patterns.

**Group-averaged cross-projection (SpontFB <-> Task):**
- R-squared 0.87-0.94: the SpontFB basis captures 87-94% of task neural variance for both DA and GABA. Strong evidence that movement direction dominates the latent structure.
- DA asymmetry: Task->Spont R-squared (~0.87) is lower than Spont->Task (~0.92-0.94), indicating ~8-13% additional task-specific variance in DA (potentially tone/CS response or reward expectation). GABA is more symmetric (~0.93 both ways) with less task-specific variance.
- **Caveat:** Group-averaged projection uses only 4 pseudo-neurons (one mean per group), compressing dimensionality. The null model effect size contextualises this.

**Null models:** All p-values confirm that observed R-squared values significantly exceed chance. Report effect sizes (observed - null_mean) / null_std for quantitative comparison.

**Scientific implication:** Movement direction is the PRIMARY organiser of DA and GABA population dynamics. The ~10% additional task variance in DA (vs GABA) may represent the value/expectation component.

---
# H2: DA and GABA Encode Different Latent Variables

**Question:** Do DA and GABA populations occupy different subspaces with distinct dimensionality and direction-selectivity?

**Prediction:** GABA is low-dimensional and strongly direction-selective. DA is high-dimensional and heterogeneous, with task-specific modulation (especially by the CS).

### H2.1: Scree Plot Comparison — DA vs GABA Eigenvalue Spectra

In [ ]:
# Fit PCA with more components to see full scree
scree_data = {}
for key in ['SpontFB_Dopamine', 'SpontFB_GABA', 'CRFB_GABA', 'ToneFB_GABA',
            'CRFB_Dopamine', 'ToneFB_Dopamine']:
    if key in results:
        r = results[key]
        n_comp = min(20, r['n_neurons'] - 1)
        pca_full = fit_pca(r['X'], n_comp)
        scree_data[key] = pca_full.explained_variance_ratio_

fig_scree = plot_scree_comparison(scree_data, title="Scree Plot: DA vs GABA")
plt.show()

### H2.2: Trajectory Metrics Comparison

In [ ]:
# Collect metrics for all successful results
all_metrics = {k: v['metrics'] for k, v in results.items()}

fig_metrics = plot_metric_comparison_table(all_metrics)
plt.show()

rows = []
for k, m in all_metrics.items():
    rows.append({
        'Analysis': k,
        'Fwd Arc Length': f"{m['fwd_arc_length']:.2f}",
        'Bwd Arc Length': f"{m['bwd_arc_length']:.2f}",
        'Mean Separation': f"{m['mean_separation']:.3f}",
        'Fwd Peak Speed': f"{np.max(m['fwd_speed']):.1f}",
        'Bwd Peak Speed': f"{np.max(m['bwd_speed']):.1f}",
    })
display(pd.DataFrame(rows))

### H2.3: Participation Ratio (Effective Dimensionality)

PR = (sum of eigenvalues)^2 / sum(eigenvalues^2). PR=1 means all variance in one dimension; PR=N means uniform distribution.

In [ ]:
# Participation ratio for all 9 analyses
pr_results = {}
for key, r in results.items():
    pr, lambdas = compute_participation_ratio(r['X'])
    pr_results[key] = pr
    print(f"{key:30s}  PR = {pr:.2f}")

fig_pr = plot_participation_ratio_comparison(pr_results,
    title="Participation Ratio: Effective Dimensionality (DA vs GABA vs Combined)")
plt.show()

# Per-epoch PR for ToneFB DA to check if reward expands dimensionality
print("\n--- ToneFB Epoch PRs ---")
tone_data = load_dataset('dataToneFB.mat', 'dataToneFB')
for ep_name, ep_cfg in EPOCHS.items():
    if ep_cfg['dataset'] not in ('ToneFB', 'Any'):
        continue
    try:
        ep_r = analyze_epoch(tone_data, DA_GROUPS, 'ToneFB', 'Dopamine',
                             ep_name, ep_cfg['start'], ep_cfg['end'],
                             timesteps=None, n_components=N_COMPONENTS, dt=DT)
        pr_ep, _ = compute_participation_ratio(ep_r['X_epoch'])
        print(f"  {ep_name:20s}  PR = {pr_ep:.2f}")
    except Exception as e:
        print(f"  {ep_name:20s}  FAILED: {e}")

### H2.4: Null Model — Fwd-Bwd Separation Significance

Phase-randomise each PC's projected timecourse independently for both forward and backward conditions, then recompute post-event mean separation. Tests whether observed separation exceeds chance given the noise structure.

In [ ]:
# Null model for fwd-bwd trajectory separation
null_sep_results = {}
for key, r in results.items():
    try:
        obs_sep, null_seps, p_val = null_separation(
            r['smooth_data'], r['window_data'], dt=DT,
            n_permutations=500, seed=42,
        )
        null_sep_results[key] = {'observed': obs_sep, 'null': null_seps, 'p': p_val}
        print(f'{key}:  sep={obs_sep:.4f}  null median={np.median(null_seps):.4f}'
              f'  p={p_val:.4f}')
    except Exception as e:
        print(f'{key}: null separation failed - {e}')

if null_sep_results:
    rows = []
    for k, res in sorted(null_sep_results.items()):
        rows.append({
            'Analysis': k,
            'Observed Sep': f"{res['observed']:.4f}",
            'Null Median': f"{np.median(res['null']):.4f}",
            'Null 95th': f"{np.percentile(res['null'], 95):.4f}",
            'p-value': f"{res['p']:.4f}",
            'Significant': 'Yes' if res['p'] < 0.05 else 'No',
        })
    display(pd.DataFrame(rows))

### H2.5: Per-Neuron-Class PCA

Run PCA on individual neuron classes (DF, DB, D, DFB, GF, GB, G, GFB) to understand what each class contributes.

**Prediction:** DF and GF should show forward-biased PC excursions; DB and GB backward-biased. Non-selective (D, G) and bidirectional (DFB, GFB) should show less separation.

In [ ]:
# Per-neuron-class PCA: run each class individually on all 3 datasets
single_class_results = {}

for ds_name, ds_cfg in DATASETS.items():
    for group in ALL_GROUPS:
        key = f"{ds_name}_{group}"
        try:
            r = analyze_dataset(
                mat_file=ds_cfg['mat_file'],
                var_name=ds_cfg['var_name'],
                dataset_name=ds_name,
                neuron_groups=[group],
                combo_label=group,
                n_components=N_COMPONENTS,
                event_idx=EVENT_IDX,
                window=WINDOW,
                dt=DT,
                sg_window=SG_WINDOW,
                sg_order=SG_ORDER,
            )
            single_class_results[key] = r
            evr = r['explained_variance_ratio']
            evr_str = '+'.join(f'{v:.3f}' for v in evr)
            sep = r['metrics']['mean_separation']
            print(f"OK  {key:25s}  n={r['n_neurons']:4d}  EVR=[{evr_str}]  sep={sep:.2f}")
        except Exception as e:
            print(f"SKIP {key:25s}  {e}")

print(f"\nSuccessful single-class analyses: {len(single_class_results)}")

In [ ]:
# Cross-class projection: fit PCA on class A, project class B
selective_results = {}

for class_a, class_b in [('DF', 'DB'), ('GF', 'GB')]:
    key_a = f'SpontFB_{class_a}'
    key_b = f'SpontFB_{class_b}'
    if key_a not in single_class_results or key_b not in single_class_results:
        print(f'SKIP {class_a}->{class_b}: missing single-class result')
        continue
    try:
        r = cross_class_project(
            result_a=single_class_results[key_a],
            result_b=single_class_results[key_b],
            window=WINDOW, event_idx=EVENT_IDX, dt=DT,
            sg_window=SG_WINDOW, sg_order=SG_ORDER,
        )
        selective_results[f'{class_a}->{class_b}'] = r
        r2_str = '  '.join(f'PC{k+1}={v:.3f}' for k, v in enumerate(r['r2_per_pc']))
        print(f'OK  {class_a}->{class_b}  mean R2={r["r2_train"]:.4f}   {r2_str}')
    except Exception as e:
        print(f'SKIP {class_a}->{class_b}: {e}')

In [ ]:
# Null model: cross-class projection R-squared (neuron identity shuffle)
null_cross_class_results = {}

for label, r in selective_results.items():
    class_a, class_b = label.split('->')
    key_a = f'SpontFB_{class_a}'
    key_b = f'SpontFB_{class_b}'
    if key_a not in single_class_results or key_b not in single_class_results:
        continue
    try:
        obs_r2, null_r2s, p_val = null_cross_class_r2(
            single_class_results[key_a], single_class_results[key_b],
            n_permutations=200, n_folds=5, seed=42,
        )
        null_cross_class_results[label] = {
            'observed': obs_r2, 'null': null_r2s, 'p': p_val}
        print(f'{label}:  CV R2={obs_r2:.4f}  null median={np.median(null_r2s):.4f}'
              f'  p={p_val:.4f}')
    except Exception as e:
        print(f'{label}: null model failed - {e}')

if null_cross_class_results:
    fig, axes = plt.subplots(1, len(null_cross_class_results),
                             figsize=(6*len(null_cross_class_results), 4))
    if len(null_cross_class_results) == 1: axes = [axes]
    for ax, (label, res) in zip(axes, null_cross_class_results.items()):
        ax.hist(res['null'], bins=30, alpha=0.7, color='grey', label='Null')
        ax.axvline(res['observed'], color='red', lw=2,
                   label=f'Observed ({res["observed"]:.3f})')
        ax.set_title(f'{label}\np = {res["p"]:.4f}')
        ax.set_xlabel('CV R-squared'); ax.legend()
    fig.suptitle('Null: Cross-Class Projection R-squared', fontweight='bold')
    plt.tight_layout(); plt.show()

In [ ]:
# Overlay: DF native + DB projected (and GF/GB)
for class_a, class_b in [('DF', 'DB'), ('GF', 'GB')]:
    key_native = f"SpontFB_{class_a}"
    key_cross = f"{class_a}->{class_b}"
    if key_native not in single_class_results or key_cross not in selective_results:
        continue
    rn = single_class_results[key_native]
    rc = selective_results[key_cross]
    tsets = [
        {
            'fwd_smooth': rn['smooth_data']['fwd_smooth'],
            'bwd_smooth': rn['smooth_data']['bwd_smooth'],
            'label': f'{class_a} (native, own space)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': rn['event_markers'],
        },
        {
            'fwd_smooth': rc['smooth_data']['fwd_smooth'],
            'bwd_smooth': rc['smooth_data']['bwd_smooth'],
            'label': f'{class_b} (projected into {class_a} space)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': rc['event_markers'],
        },
    ]
    fig = build_overlay_figure(
        tsets, title=f'H2: {class_b} onto {class_a} PC space (SpontFB)')
    fig.show()

In [ ]:
# Separation and PC1 EVR across neuron classes (SpontFB)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

classes = ALL_GROUPS
sep_vals, evr_vals = [], []
for g in classes:
    key = f"SpontFB_{g}"
    if key in single_class_results:
        r = single_class_results[key]
        sep_vals.append(r['metrics']['mean_separation'])
        evr_vals.append(r['explained_variance_ratio'][0])
    else:
        sep_vals.append(0); evr_vals.append(0)

colors = ['#e74c3c', '#c0392b', '#e67e22', '#d35400',
          '#3498db', '#2980b9', '#1abc9c', '#16a085']
x = np.arange(len(classes))

axes[0].bar(x, sep_vals, color=colors, alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(classes, fontsize=11)
axes[0].set_ylabel('Mean Fwd-Bwd Separation', fontsize=12)
axes[0].set_title('SpontFB: Fwd-Bwd Separation per Neuron Class', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x, evr_vals, color=colors, alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(classes, fontsize=11)
axes[1].set_ylabel('PC1 Explained Variance Ratio', fontsize=12)
axes[1].set_title('SpontFB: PC1 EVR per Neuron Class', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

fig.suptitle('H2: Per-Neuron-Class Analysis (SpontFB)', fontsize=14, y=1.02)
fig.tight_layout(); plt.show()

### H2.6: PC Loadings by Neuron Group

In [ ]:
# PC loading analysis for all 9 main analyses
for key in sorted(results.keys()):
    r = results[key]
    groups = r['config']['neuron_groups']
    try:
        loadings_df = compute_pc_loadings_by_group(r['pca'], groups, r['stats'])
        fig_load = plot_pc_loadings(loadings_df, title=f"PC Loadings: {key}")
        plt.show()
        display(loadings_df.round(4))
    except Exception as e:
        print(f"{key}: failed -- {e}")

### H2.7: RSA + Procrustes — DA vs GABA Manifold Comparison

In [ ]:
# RSA: DA vs GABA (epoch-based with rsatoolbox inference)
rsa_h2 = compute_rsa(results, pop_a='Dopamine', pop_b='GABA',
                      method='cosine', n_bootstrap=1000)

print("Per-epoch DA vs GABA RDM similarity (cosine):")
for name, info in rsa_h2['per_epoch'].items():
    print(f"  {name:20s}  sim={info['similarity']:.4f}  "
          f"({info['dataset']}, {info['n_timepoints']} tp)  {info['epoch_desc']}")

t = rsa_h2['ttest']
print(f"\nOne-sample t-test (H0: similarity=0): "
      f"t({t['df']})={t['t']:.3f}, p={t['p']:.4e}")
print(f"Mean similarity: {rsa_h2['mean']:.4f} +/- {rsa_h2['sem']:.4f} (SEM)")

for n_tp, mi in rsa_h2['model_inference'].items():
    print(f"\n--- rsatoolbox inference (n_timepoints={n_tp}) ---")
    print(f"  Epochs: {mi['group_names']}")
    evals = mi['eval_fixed'].evaluations.squeeze()
    print(f"  eval_fixed evaluations: {evals}")
    if mi['eval_bootstrap'] is not None:
        means = mi['eval_bootstrap'].get_means().squeeze()
        sems = mi['eval_bootstrap'].get_sem().squeeze()
        print(f"  Bootstrap mean: {means:.4f}, SEM: {sems:.4f}")

fig = plot_rsa_comparison(rsa_h2, title='H2.7: RSA -- DA vs GABA (epoch-based)')
plt.show()

# Procrustes: DA vs GABA smoothed trajectories
procrustes_h2_results = {}
for ds_name in ['SpontFB', 'CRFB', 'ToneFB']:
    da_key = f'{ds_name}_Dopamine'
    ga_key = f'{ds_name}_GABA'
    if da_key not in results or ga_key not in results:
        continue
    sd_da = results[da_key]['smooth_data']
    sd_ga = results[ga_key]['smooth_data']
    proc = compute_procrustes_comparison(sd_da, sd_ga)
    procrustes_h2_results[ds_name] = proc
    for d, v in proc.items():
        print(f"  Procrustes {ds_name} {d}: disparity={v['disparity']:.4f}")

if 'SpontFB_Dopamine' in results and 'SpontFB_GABA' in results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    n_t = results['SpontFB_Dopamine']['X'].shape[1] // 2
    rdm_da = compute_rdm(results['SpontFB_Dopamine']['X'].values[:, :n_t])
    rdm_ga = compute_rdm(results['SpontFB_GABA']['X'].values[:, :n_t])
    vmin = min(rdm_da.min(), rdm_ga.min())
    vmax = max(rdm_da.max(), rdm_ga.max())
    for ax, rdm, lbl in [(axes[0], rdm_da, 'DA (730 neurons)'),
                          (axes[1], rdm_ga, 'GABA (102 neurons)')]:
        im = ax.imshow(rdm, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(f'SpontFB {lbl} -- fwd RDM')
        ax.set_xlabel('Timepoint'); ax.set_ylabel('Timepoint')
    fig.colorbar(im, ax=axes, shrink=0.8, label='1 - Pearson r')
    plt.tight_layout(); plt.show()

if procrustes_h2_results:
    flat_proc = {}
    for ds, dirs in procrustes_h2_results.items():
        for d, v in dirs.items():
            flat_proc[f'{ds}_{d}'] = v
    plot_procrustes_comparison(flat_proc, title='H2.7: Procrustes -- DA vs GABA')

### H2 — Interpretation

**Dimensionality (scree plots + participation ratio):**
- GABA has a steep scree drop: one or two PCs dominate, then rapid fall-off. GABA population activity is low-rank / coordinated, consistent with a homogeneous inhibitory signal.
- DA has a flat scree: variance distributed across many PCs. DA population is heterogeneous with diverse subpopulations (DF, DB, D, DFB) with distinct tuning.
- ToneFB DA scree is steeper than SpontFB DA or CRFB DA (PC1 = 20.9% vs 7-9%). The tone synchronises DA subpopulations more strongly than movement onset alone.
- Combined (DA+GABA) participation ratio lies between pure-DA and pure-GABA values.

**Trajectory metrics:**
- GABA (all datasets): high mean separation (19-28) -- fwd and bwd trajectories occupy clearly different regions. GABA is strongly direction-selective at the population level.
- DA SpontFB: lower separation (12.2) than GABA. DA is less direction-selective in spontaneous movement.
- ToneFB DA has the largest arc lengths (fwd ~1095, bwd ~1064) of any analysis, but with less fwd-bwd separation (7.8) than GABA. DA doesn't strongly separate movement direction during task; instead it shows large absolute excursions driven by the CS.

**Per-neuron-class PCA:**
- DF/GF should show highest forward-biased separation; DB/GB highest backward-biased separation.
- Cross-class projection: if DB-backward trajectory in DF PC space resembles DF-forward trajectory, both populations encode the same direction variable at the population level.

**Null models:** Fwd-bwd separation and cross-class R-squared values both significantly exceed null distributions.

**Implication:** DA and GABA encode qualitatively different variables. GABA's high dimensionality-normalised separation suggests directional encoding. DA's large amplitude with poor direction separation suggests CS/value-driven encoding.

---
# H3: Is There a Reward-Specific Signal in GABA/DA Trajectories?

**Question:** If GABA carries value information, trajectories should show a deflection at reward delivery (+1.0s in ToneFB). SpontFB serves as the null control (no reward at +1.0s).

**Prediction (movement theory):** No reward-specific deflection in GABA or DA; speed profiles should be similar at equivalent post-event latency in ToneFB vs SpontFB.

**Prediction (RPE theory):** A sharp excursion at reward delivery in one or more PCs, absent in SpontFB.

### H3.1: 1D PC Timecourses — GABA (ToneFB vs SpontFB)

In [ ]:
if 'ToneFB_GABA' in results:
    r = results['ToneFB_GABA']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='ToneFB GABA -- 1D PC Timecourses (reward at +1.0s)',
        dataset_name='ToneFB',
    )
    plt.show()
else:
    print('ToneFB_GABA not available')

In [ ]:
if 'SpontFB_GABA' in results:
    r = results['SpontFB_GABA']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='SpontFB GABA -- 1D PC Timecourses (null: no reward)',
        dataset_name='SpontFB',
    )
    plt.show()
else:
    print('SpontFB_GABA not available')

### H3.2: 1D PC Timecourses — Dopamine (ToneFB vs SpontFB)

In [ ]:
if 'ToneFB_Dopamine' in results:
    r = results['ToneFB_Dopamine']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='ToneFB Dopamine -- 1D PC Timecourses',
        dataset_name='ToneFB',
    )
    plt.show()
else:
    print('ToneFB_Dopamine not available')

In [ ]:
if 'SpontFB_Dopamine' in results:
    r = results['SpontFB_Dopamine']
    fig_1d = plot_1d_pc_timecourses(
        r['window_data'], r['smooth_data'], r['event_markers'],
        title='SpontFB Dopamine -- 1D PC Timecourses (null: no reward)',
        dataset_name='SpontFB',
    )
    plt.show()
else:
    print('SpontFB_Dopamine not available')

### H3.3: Speed Profiles — ToneFB vs SpontFB

In [ ]:
# Speed profiles: GABA
speed_metrics = {}
for key in ['ToneFB_GABA', 'SpontFB_GABA']:
    if key in results:
        speed_metrics[key] = results[key]['metrics']
if speed_metrics:
    fig_speed = plot_speed_profiles(speed_metrics, title='Speed Profiles: ToneFB vs SpontFB GABA')
    plt.show()

In [ ]:
# Speed profiles: Dopamine
speed_metrics = {}
for key in ['ToneFB_Dopamine', 'SpontFB_Dopamine']:
    if key in results:
        speed_metrics[key] = results[key]['metrics']
if speed_metrics:
    fig_speed = plot_speed_profiles(speed_metrics, title='Speed Profiles: ToneFB vs SpontFB Dopamine')
    plt.show()

### H3.4: Trajectory Divergence Onset

When do fwd and bwd trajectories first separate? Compare across datasets and neuron types.

**Prediction:** Divergence starts at or near movement onset (0 s), consistently across all datasets.

In [ ]:
# Divergence onset for all 9 analyses
div_dict = {}
for key, r in results.items():
    onset_t, onset_i, sep, thresh = compute_divergence_onset(
        r['smooth_data'], r['window_data'], dt=DT,
        baseline_window=20, threshold_factor=2.0,
    )
    div_dict[key] = (onset_t, onset_i, sep, thresh, r['window_data']['plot_time'])
    if onset_t is not None:
        print(f"{key:30s}  onset = {onset_t:+.3f} s")
    else:
        print(f"{key:30s}  onset = never (sep never 2x baseline)")

gaba_div = {k: v for k, v in div_dict.items() if 'GABA' in k}
da_div   = {k: v for k, v in div_dict.items() if 'Dopamine' in k}

fig_div_gaba = plot_divergence_comparison(gaba_div, title="Fwd-Bwd Divergence Onset: GABA")
plt.show()
fig_div_da = plot_divergence_comparison(da_div, title="Fwd-Bwd Divergence Onset: Dopamine")
plt.show()

### H3.5: Cross-Epoch PCA R-squared Heatmap

Fit PCA on one epoch, project another. If pre-tone and post-reward epochs share high R-squared, the same latent structure persists across task phases (supporting movement encoding). If they diverge, reward changes the encoding.

In [ ]:
# Cross-epoch R-squared matrix for all datasets
for key, r in results.items():
    ds_name = r['config']['dataset_name']
    relevant_epochs = {}
    for ename, ecfg in EPOCHS.items():
        if ecfg['dataset'] == ds_name or ecfg['dataset'] == 'Any':
            if ecfg['end'] <= r['timesteps']:
                relevant_epochs[ename] = ecfg
    if len(relevant_epochs) < 2:
        continue

    print(f"\n=== {key} ===")
    print(f"Epochs: {list(relevant_epochs.keys())}")

    r2_df, pca_dict = compute_cross_epoch_r2_matrix(
        r['X'], r['timesteps'], relevant_epochs,
        neuron_groups_label=r['config']['combo_label'],
        n_components=N_COMPONENTS,
    )
    fig = plot_cross_epoch_r2_matrix(r2_df, title=f'Cross-Epoch R-squared: {key}')
    plt.show()
    display(r2_df.round(3))

### H3.6: Null Model — Reward-Time Deflection Test

**Within-dataset test:** Is speed at reward time (+1.0s in ToneFB) significantly higher than at other post-event timepoints? Uses circular time-shift null.

**Between-dataset test:** Is speed at ToneFB[+1.0s] significantly different from SpontFB at equivalent post-event latency? Uses permutation test on pooled post-event speeds.

In [ ]:
# Null model for reward-time deflection: GABA
null_reward_gaba = None
if 'ToneFB_GABA' in results and 'SpontFB_GABA' in results:
    null_reward_gaba = null_reward_deflection(
        smooth_data_task=results['ToneFB_GABA']['smooth_data'],
        window_data_task=results['ToneFB_GABA']['window_data'],
        smooth_data_ctrl=results['SpontFB_GABA']['smooth_data'],
        window_data_ctrl=results['SpontFB_GABA']['window_data'],
        reward_offset=100, dt=DT,
        test_half_width=10, n_permutations=1000, seed=42,
    )
    wt = null_reward_gaba['within_task']
    bt = null_reward_gaba['between_datasets']
    print("=== GABA Reward Deflection Test ===")
    print(f"  Within ToneFB: speed at reward = {wt['observed_speed']:.2f}, "
          f"p = {wt['p_value']:.4f}")
    print(f"  ToneFB vs SpontFB: diff = {bt['observed_diff']:.2f}, "
          f"p = {bt['p_value']:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(wt['null_speeds'], bins=30, alpha=0.7, color='grey', label='Null')
    axes[0].axvline(wt['observed_speed'], color='red', lw=2,
                    label=f'Observed ({wt["observed_speed"]:.2f})')
    axes[0].set_title(f'GABA Within-ToneFB\np = {wt["p_value"]:.4f}')
    axes[0].set_xlabel('Speed at reward time'); axes[0].legend()

    axes[1].hist(bt['null_diffs'], bins=30, alpha=0.7, color='grey', label='Null')
    axes[1].axvline(bt['observed_diff'], color='red', lw=2,
                    label=f'Observed ({bt["observed_diff"]:.2f})')
    axes[1].set_title(f'GABA ToneFB vs SpontFB\np = {bt["p_value"]:.4f}')
    axes[1].set_xlabel('Speed difference'); axes[1].legend()
    fig.suptitle('Null: GABA Reward-Time Deflection', fontweight='bold')
    plt.tight_layout(); plt.show()

# Null model for reward-time deflection: Dopamine
null_reward_da = None
if 'ToneFB_Dopamine' in results and 'SpontFB_Dopamine' in results:
    null_reward_da = null_reward_deflection(
        smooth_data_task=results['ToneFB_Dopamine']['smooth_data'],
        window_data_task=results['ToneFB_Dopamine']['window_data'],
        smooth_data_ctrl=results['SpontFB_Dopamine']['smooth_data'],
        window_data_ctrl=results['SpontFB_Dopamine']['window_data'],
        reward_offset=100, dt=DT,
        test_half_width=10, n_permutations=1000, seed=42,
    )
    wt = null_reward_da['within_task']
    bt = null_reward_da['between_datasets']
    print("\n=== DA Reward Deflection Test ===")
    print(f"  Within ToneFB: speed at reward = {wt['observed_speed']:.2f}, "
          f"p = {wt['p_value']:.4f}")
    print(f"  ToneFB vs SpontFB: diff = {bt['observed_diff']:.2f}, "
          f"p = {bt['p_value']:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(wt['null_speeds'], bins=30, alpha=0.7, color='grey', label='Null')
    axes[0].axvline(wt['observed_speed'], color='red', lw=2,
                    label=f'Observed ({wt["observed_speed"]:.2f})')
    axes[0].set_title(f'DA Within-ToneFB\np = {wt["p_value"]:.4f}')
    axes[0].set_xlabel('Speed at reward time'); axes[0].legend()

    axes[1].hist(bt['null_diffs'], bins=30, alpha=0.7, color='grey', label='Null')
    axes[1].axvline(bt['observed_diff'], color='red', lw=2,
                    label=f'Observed ({bt["observed_diff"]:.2f})')
    axes[1].set_title(f'DA ToneFB vs SpontFB\np = {bt["p_value"]:.4f}')
    axes[1].set_xlabel('Speed difference'); axes[1].legend()
    fig.suptitle('Null: DA Reward-Time Deflection', fontweight='bold')
    plt.tight_layout(); plt.show()

### H3 — Interpretation

**1D PC timecourses:**
- Examine ToneFB GABA PCs at the reward marker (+1.0s). If a reward-specific deflection is present in ToneFB but absent in SpontFB at the equivalent latency, this supports value encoding. If both datasets show similar dynamics at matched post-event latency, the signal is movement-driven.

**Speed profiles:**
- Compare speed transients at +1.0s in ToneFB vs the equivalent time in SpontFB. A reward-locked speed transient unique to ToneFB would support value encoding.

**Divergence onset:**
- If fwd-bwd divergence onset is consistent across SpontFB, CRFB, and ToneFB (all near movement onset), this supports movement as the primary driver regardless of task context.

**Cross-epoch PCA:**
- High R-squared between pre-tone and post-reward epochs means the same latent structure persists across task phases, supporting movement encoding. Low R-squared means reward fundamentally changes the encoding.

**Null model results:**
- Within-ToneFB p-value tests whether speed at reward time is significantly higher than at other post-event timepoints. Between-dataset p-value tests whether ToneFB and SpontFB differ at matched latency.
- If neither test is significant: no evidence for reward-specific encoding.

**Conclusion:** [To be filled based on observed results — if null model p-values are non-significant, conclude that the evidence does not support reward-specific encoding in GABA or DA trajectories. If significant, the nature and magnitude of the deflection should be characterised.]

---
# H4: Are CS-Evoked and CR-Evoked Phasic DA the Same Signal Time-Shifted?

**Question:** ToneFB (tone-aligned) and CRFB (movement-aligned) share the same neurons. If the phasic DA burst at CS-onset and the burst at CR-onset are the same movement-related event appearing at different latencies, then:
1. Cross-correlation of PC timecourses should peak at the reaction-time lag (~0.1-0.4s)
2. Procrustes disparity should be low (similar trajectory shape)
3. RSA should show high temporal RDM similarity

If CS and CR produce genuinely different dynamics (e.g., value vs motor), cross-correlation should not show a clean lag peak, and RSA/Procrustes should indicate dissimilar geometry.

**Note:** Cross-projection R-squared is reported in H1.1 above. This section focuses on the temporal and geometric comparison.

### H4.1: Cross-Correlation of PC Timecourses (ToneFB vs CRFB)

Cross-correlate each PC's smoothed timecourse between ToneFB and CRFB. PCA axes are aligned before correlating.

In [ ]:
# Cross-correlation: ToneFB vs CRFB
for pop_name, pop_label in [('GABA', 'GABA'), ('Dopamine', 'DA')]:
    tone_key = f'ToneFB_{pop_name}'
    cr_key = f'CRFB_{pop_name}'
    if tone_key not in results or cr_key not in results:
        continue

    # Align CRFB PCA axes to ToneFB before cross-correlating
    import copy
    pca_cr_aligned = copy.deepcopy(results[cr_key]['pca'])
    pca_cr_aligned, _ = align_pca_signs(pca_cr_aligned, results[tone_key]['pca'])

    # Re-project CRFB onto aligned axes
    proj_cr_aligned = project_onto_pca(pca_cr_aligned, results[cr_key]['X'])
    cr_window = slice_window(proj_cr_aligned, results[cr_key]['timesteps'],
                             event_idx=EVENT_IDX, window=WINDOW, dt=DT)
    cr_fwd_smooth = smooth_trajectories(cr_window['fwd'], SG_WINDOW, SG_ORDER)
    cr_bwd_smooth = smooth_trajectories(cr_window['bwd'], SG_WINDOW, SG_ORDER)

    fig, axes_arr = plt.subplots(N_COMPONENTS, 2, figsize=(14, 3*N_COMPONENTS),
                                 sharex=True)
    directions = ['fwd_smooth', 'bwd_smooth']
    dir_labels = ['Forward', 'Backward']

    for col, (direction, dlabel) in enumerate(zip(directions, dir_labels)):
        tone_smooth = results[tone_key]['smooth_data'][direction]
        cr_smooth = cr_fwd_smooth if direction == 'fwd_smooth' else cr_bwd_smooth

        for pc in range(N_COMPONENTS):
            lags, corr = compute_cross_correlation(
                tone_smooth[pc], cr_smooth[pc], dt=DT)
            axes_arr[pc, col].plot(lags, corr, color='purple', linewidth=1.5)
            peak_lag = lags[np.argmax(corr)]
            axes_arr[pc, col].axvline(peak_lag, color='red', linestyle='--',
                                      alpha=0.7, label=f'peak={peak_lag:.2f}s')
            axes_arr[pc, col].axvline(0, color='grey', linestyle=':', alpha=0.5)
            axes_arr[pc, col].set_ylabel(f'PC{pc+1}')
            axes_arr[pc, col].legend(fontsize=8)
            axes_arr[pc, col].grid(True, alpha=0.3)
            if pc == 0:
                axes_arr[pc, col].set_title(f'{dlabel}', fontsize=12)

    axes_arr[-1, 0].set_xlabel('Lag (s)')
    axes_arr[-1, 1].set_xlabel('Lag (s)')
    fig.suptitle(f'H4: Cross-Correlation ToneFB vs CRFB {pop_label} (aligned axes)',
                 fontsize=14, y=1.02)
    fig.tight_layout(); plt.show()

### H4.2: Overlay — ToneFB vs CRFB Dynamics

In [ ]:
# Overlay: ToneFB native + CRFB projected for DA
if 'DA Tone->CR' in cross_results_same_neuron and 'ToneFB_Dopamine' in results:
    tsets = [
        {
            'fwd_smooth': results['ToneFB_Dopamine']['smooth_data']['fwd_smooth'],
            'bwd_smooth': results['ToneFB_Dopamine']['smooth_data']['bwd_smooth'],
            'label': 'ToneFB DA (native)',
            'fwd_color': 'orangered', 'bwd_color': 'royalblue', 'dash': 'solid',
            'event_markers': results['ToneFB_Dopamine']['event_markers'],
        },
        {
            'fwd_smooth': cross_results_same_neuron['DA Tone->CR']['smooth_data']['fwd_smooth'],
            'bwd_smooth': cross_results_same_neuron['DA Tone->CR']['smooth_data']['bwd_smooth'],
            'label': 'CRFB DA (projected)',
            'fwd_color': 'darkorange', 'bwd_color': 'steelblue', 'dash': 'dash',
            'event_markers': cross_results_same_neuron['DA Tone->CR']['event_markers'],
        },
    ]
    fig_overlay = build_overlay_figure(
        tsets, title='H4: ToneFB vs CRFB Dopamine (ToneFB PC space)')
    fig_overlay.show()

### H4.3: RSA + Procrustes — ToneFB vs CRFB (Same Neurons)

Dual RSA: computed on both full neural space and 3-PC projections.

In [ ]:
# RSA + Procrustes: ToneFB vs CRFB
rsa_h4_results = {}
procrustes_h4_results = {}

for combo_name in ['GABA', 'Dopamine']:
    tone_key = f'ToneFB_{combo_name}'
    cr_key = f'CRFB_{combo_name}'
    if tone_key not in results or cr_key not in results:
        continue

    X_tone = results[tone_key]['X']
    X_cr = results[cr_key]['X']
    n_t = X_tone.shape[1] // 2

    # Full-space RSA
    for half_name, sl in [('fwd', slice(0, n_t)), ('bwd', slice(n_t, 2*n_t))]:
        rdm_tone = compute_rdm(X_tone.values[:, sl])
        rdm_cr = compute_rdm(X_cr.values[:, sl])
        r_val = compare_rdms(rdm_tone, rdm_cr, method='cosine')
        label = f'{combo_name}_{half_name} (full-space)'
        rsa_h4_results[label] = {'r': r_val, 'p': float('nan')}
        print(f"  RSA {label}: similarity={r_val:.4f}")

    # 3-PC RSA
    proj_tone = results[tone_key]['projections']
    proj_cr = results[cr_key]['projections']
    for half_name, sl in [('fwd', slice(0, n_t)), ('bwd', slice(n_t, 2*n_t))]:
        rdm_tone_pc = compute_rdm(proj_tone[:, sl])
        rdm_cr_pc = compute_rdm(proj_cr[:, sl])
        r_val = compare_rdms(rdm_tone_pc, rdm_cr_pc, method='cosine')
        label = f'{combo_name}_{half_name} (3-PC)'
        rsa_h4_results[label] = {'r': r_val, 'p': float('nan')}
        print(f"  RSA {label}: similarity={r_val:.4f}")

    # Procrustes
    sd_tone = results[tone_key]['smooth_data']
    sd_cr = results[cr_key]['smooth_data']
    proc = compute_procrustes_comparison(sd_tone, sd_cr)
    procrustes_h4_results[combo_name] = proc
    for d, v in proc.items():
        print(f"  Procrustes {combo_name} {d}: disparity={v['disparity']:.4f}")

# RDM heatmaps for GABA
if 'ToneFB_GABA' in results and 'CRFB_GABA' in results:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    n_t = results['ToneFB_GABA']['X'].shape[1] // 2
    rdm_tone = compute_rdm(results['ToneFB_GABA']['X'].values[:, :n_t])
    rdm_cr = compute_rdm(results['CRFB_GABA']['X'].values[:, :n_t])
    vmin = min(rdm_tone.min(), rdm_cr.min())
    vmax = max(rdm_tone.max(), rdm_cr.max())
    for ax, rdm, lbl in [(axes[0], rdm_tone, 'ToneFB'), (axes[1], rdm_cr, 'CRFB')]:
        im = ax.imshow(rdm, cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(f'GABA {lbl} -- fwd RDM')
        ax.set_xlabel('Timepoint'); ax.set_ylabel('Timepoint')
    fig.colorbar(im, ax=axes, shrink=0.8, label='1 - Pearson r')
    plt.tight_layout(); plt.show()

if rsa_h4_results:
    plot_rsa_comparison(rsa_h4_results, title='H4.3: RSA -- ToneFB vs CRFB')
if procrustes_h4_results:
    flat_proc = {}
    for combo, dirs in procrustes_h4_results.items():
        for d, v in dirs.items():
            flat_proc[f'{combo}_{d}'] = v
    plot_procrustes_comparison(flat_proc, title='H4.3: Procrustes -- ToneFB vs CRFB')

### H4 — Interpretation

**Cross-correlation:**
- Peak lag at approximately reaction time (~0.1-0.4s): the CS burst and CR burst are the same event with a delay. Symmetric peaks in both directions confirm this.
- Consistent with a single movement-related burst appearing at different phases of each alignment window.

**Subspace overlap (from H1.1):**
- GABA: [0.985, 0.982, 0.927] — principal angles nearly 0 degrees. ToneFB and CRFB GABA span essentially the same 3D subspace. The axes of population variability are preserved across alignment conditions.
- DA: partial overlap — some shared dimensions but 3rd PC may encode a different latent variable.

**Reconstruction R-squared (from H1.1):**
- GABA R-squared ~0.40-0.45: same subspace but only ~40% of temporal variance reconstructed. Same latent axes, different temporal dynamics. CS-onset and CR-onset modulate the same population modes with different temporal profiles.
- DA R-squared ~0.08-0.18: much lower — DA dynamics are more timing-sensitive; CS and CR activate different temporal patterns even in the same basis.

**Procrustes disparity:**
- Low disparity = similar trajectory shape (after removing rotation and scale). Complements R-squared which captures temporal alignment.

**RSA (full-space vs 3-PC):**
- If full-space RSA is high but 3-PC RSA is low: shared structure exists beyond the top 3 PCs. If both are high: the top 3 PCs capture the shared temporal geometry.

**Conclusion:** CS and CR events engage the same population modes with different temporal profiles. This is inconsistent with "CS = value signal, CR = motor signal in orthogonal subspaces." Instead, both reflect movement-related dynamics at different temporal phases.

---
# Summary: All Quantitative Metrics

In [ ]:
# Unified summary table
summary_rows = []

# Within-dataset metrics
for key, r in results.items():
    m = r['metrics']
    evr = r['explained_variance_ratio']
    pr, _ = compute_participation_ratio(r['X'])
    onset_t, _, _, _ = compute_divergence_onset(r['smooth_data'], r['window_data'], dt=DT)
    summary_rows.append({
        'Analysis': key, 'Type': 'Within-dataset',
        'N_neurons': r['n_neurons'],
        'PC1_var': f"{evr[0]:.3f}", 'PC2_var': f"{evr[1]:.3f}",
        'PC3_var': f"{evr[2]:.3f}", 'Total_var': f"{sum(evr):.3f}",
        'PR': f"{pr:.1f}",
        'Mean_sep': f"{m['mean_separation']:.3f}",
        'Peak_sep': f"{m.get('peak_separation', 0):.3f}",
        'PostEvt_sep': f"{m.get('post_event_mean_separation', 0):.3f}",
        'Div_onset_s': f"{onset_t:+.3f}" if onset_t is not None else 'never',
        'R2': 'N/A',
    })

# Cross-projection metrics
for label, cr_dict in [('H1 same-neuron', cross_results_same_neuron),
                        ('H1 group-avg', cross_results_group_avg)]:
    for name, cr in cr_dict.items():
        m = cr['metrics']
        summary_rows.append({
            'Analysis': f"{label}: {name}", 'Type': 'Cross-projection',
            'N_neurons': '', 'PC1_var': '', 'PC2_var': '', 'PC3_var': '',
            'Total_var': '', 'PR': '',
            'Mean_sep': f"{m['mean_separation']:.3f}",
            'Peak_sep': f"{m.get('peak_separation', 0):.3f}",
            'PostEvt_sep': f"{m.get('post_event_mean_separation', 0):.3f}",
            'Div_onset_s': '', 'R2': f"{cr['r2']:.4f}",
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## Interpretation Guide

| Hypothesis | Key Evidence | Threshold | Conclusion |
|---|---|---|---|
| **H1** | SpontFB->Task R-squared | > 0.7 = movement dominates | R-squared 0.87-0.94: movement is primary organiser |
| **H1** | Task->Spont asymmetry | DA > GABA asymmetry | ~10% additional task variance in DA |
| **H2** | GABA PC1 variance | 27-39% (vs DA 7-21%) | GABA is low-rank/coordinated |
| **H2** | Fwd-bwd separation | GABA >> DA | GABA encodes direction more strongly |
| **H3** | Deflection at reward time | Significant in null model | [Check null model p-values] |
| **H4** | Cross-correlation peak lag | ~ reaction time | Same burst, time-shifted |
| **H4** | Subspace overlap | ~ 1.0 | Same latent axes for CS and CR |

**Future (H5):** Project airpuff data onto ToneFB PCA space. If R-squared drops significantly compared to ToneFB->CRFB, aversive stimuli drive orthogonal dynamics (value axis exists). If R-squared remains high, aversive stimuli activate the same movement-direction subspace.

## Conclusion and Future Directions

**Summary of evidence:**
1. **H1 (Supported):** Movement direction dominates VTA latent structure. Cross-projection R-squared 0.87-0.94 across contexts; null models confirm significance.
2. **H2 (Supported):** DA and GABA encode different latent variables. GABA is low-dimensional and direction-selective; DA is heterogeneous with large CS-driven dynamics but poor direction separation.
3. **H3 (Pending results):** Whether a reward-specific deflection exists depends on the null model p-values computed above. The interpretation framework is in place.
4. **H4 (Supported):** CS-evoked and CR-evoked dynamics share the same subspace with a reaction-time lag, consistent with the same movement-related burst.

**Next steps:**
- **H5 (Airpuff data):** Test whether aversive stimuli project onto the same or orthogonal subspace.
- **Behavioral correlation:** Once per-trial force/lick data is aligned to z-scored firing rates, call `compute_pc_behavioral_correlation()` to quantitatively link PCs to kinematics.
- **Trial-level analysis:** Current analyses use trial-averaged data. Single-trial PCA would allow cross-validation across trials and strengthen claims about encoding.